# Analyse mentions of electricity costs in Parliament

In [ ]:
from discovery_utils.synthesis.policy import policy_update
from discovery_utils.utils import keywords as kw
from discovery_utils.utils.llm import batch_check

import pandas as pd
import datetime

In [3]:
HansardData = policy_update.HansardData()

2025-08-05 10:30:02,452 - discovery_utils.getters.hansard - INFO - Downloading debates parquet file: data/policy_scanning_data/enriched/HansardDebates.parquet
2025-08-05 10:32:33,471 - discovery_utils.getters.hansard - INFO - Attempting to download label store: data/policy_scanning_data/enriched/HansardDebates_LabelStore_keywords.csv
2025-08-05 10:35:17,696 - discovery_utils.getters.hansard - INFO - Downloading people metadata
2025-08-05 10:35:24,260 - discovery_utils.getters.hansard - INFO - Successfully downloaded and saved people metadata


In [126]:
def get_quarter_from_date(date:str) -> int:
    """Return the quarter number from a given YYYY-MM-DD date string."""
    _date = datetime.datetime.strptime(date, "%Y-%m-%d")
    return (_date.month-1)//3 + 1

def create_minimal_electricity_price_keywords():
    """Create keywords dictionary for electricity price/cost analysis."""
    return {
        "electricity_prices": [
            ["electricity", "price"],
            ["electricity", "cost"],
            ["electricity", "pricing"],
            ["electricity", "costs"],
            ["electricity", "bill"],
            ["electricity", "bills"],
            ["electricity", "tariff"],
            ["electricity", "tariffs"],
            ["electricity", "expensive"],
            ["electricity", "expense"],
            ["electricity", "payment"],
            ["electricity", "payments"],
            ["electricity", "spend"],
            ["electricity", "spending"],
            ["electricity", "expenditure"],
            ["electricity", "affordability"],
            ["electricity", "affordable"],
            ["electricity", "expensive"],
            ["electricity", "cheap"],
            ["electricity", "cheaper"],
            ["electricity", "costly"],
            ["electricity", "money"],
            ["electricity", "pound"],
            ["electricity", "pounds"],
        ]
    }

def analyze_electricity_prices(HansardData:policy_update.HansardData, test_mode:bool=False):
    """Main function to analyze electricity price mentions in Hansard speeches."""
    
    print("Processing speeches...")
    speeches_df = (
        HansardData.debates_df
        .query("date >= '2014-01-01' & date <= '2025-08-31'")
        .assign(quarter=lambda df: df.date.apply(get_quarter_from_date))
        .assign(quarter=lambda df: df.year.astype(str) + "-Q" + df.quarter.astype(str))
        .reset_index(drop=True)
    )
    if test_mode:
        speeches_df = speeches_df.head(100)
    
    print(f"Loaded {len(speeches_df)} speeches from 2014 to 2025")
    
    # Create keywords dictionary
    keywords_dict = create_minimal_electricity_price_keywords()
    
    # Process each speech to find electricity price mentions
    print("Searching for electricity price mentions...")
    all_hits = []
    
    for idx, row in speeches_df.iterrows():
        if idx % 10000 == 0:
            print(f"Processed {idx} speeches...")
        
        speech_text = row['speech']
        speech_id = row['speech_id']
        speaker = row['speakername']
        date = row['date']
        quarter = row['quarter']
        major_heading = row['major_heading']
        minor_heading = row['minor_heading']
        
        # Get keyword hits for this speech
        hits_df = kw.get_keyword_hits(speech_text, keywords_dict)
        
        if not hits_df.empty:
            # Add metadata to each hit
            hits_df['speech_id'] = speech_id
            hits_df['speaker'] = speaker
            hits_df['date'] = date
            hits_df['quarter'] = quarter
            hits_df['major_heading'] = major_heading
            hits_df['minor_heading'] = minor_heading
            
            all_hits.append(hits_df)
    
    if all_hits:
        # Combine all hits
        combined_hits = pd.concat(all_hits, ignore_index=True)
        
        print(f"\nFound {len(combined_hits)} sentences mentioning electricity prices/costs")
        
        # Save results
        output_file = "electricity_price_mentions.csv"
        combined_hits.to_csv(output_file, index=False)
        print(f"Results saved to {output_file}")
        
        # Display summary statistics
        print("\nSummary by quarter:")
        quarterly_summary = combined_hits.groupby('quarter').size().sort_index()
        print(quarterly_summary)
        
        print("\nTop speakers mentioning electricity prices:")
        speaker_summary = combined_hits.groupby('speaker').size().sort_values(ascending=False).head(10)
        print(speaker_summary)
        
        print("\nSample sentences:")
        sample_sentences = combined_hits[['date', 'speaker', 'sentence']].head(10)
        for _, row in sample_sentences.iterrows():
            print(f"{row['date']} - {row['speaker']}: {row['sentence'][:200]}...")
        
        return combined_hits
    else:
        print("No electricity price mentions found.")
        return pd.DataFrame()

In [127]:
hits_df = analyze_electricity_prices(HansardData, test_mode=False)

Processing speeches...
Loaded 746897 speeches from 2014 to 2025
Searching for electricity price mentions...
Processed 0 speeches...
Processed 10000 speeches...
Processed 20000 speeches...
Processed 30000 speeches...
Processed 40000 speeches...
Processed 50000 speeches...
Processed 60000 speeches...
Processed 70000 speeches...
Processed 80000 speeches...
Processed 90000 speeches...
Processed 100000 speeches...
Processed 110000 speeches...
Processed 120000 speeches...
Processed 130000 speeches...
Processed 140000 speeches...
Processed 150000 speeches...
Processed 160000 speeches...
Processed 170000 speeches...
Processed 180000 speeches...
Processed 190000 speeches...
Processed 200000 speeches...
Processed 210000 speeches...
Processed 220000 speeches...
Processed 230000 speeches...
Processed 240000 speeches...
Processed 250000 speeches...
Processed 260000 speeches...
Processed 270000 speeches...
Processed 280000 speeches...
Processed 290000 speeches...
Processed 300000 speeches...
Process

In [128]:
_hits_df = (
    hits_df
    .merge(HansardData.debates_df[["speech_id", "speech"]], on="speech_id", how="left")
)

In [130]:
_hits_df

,sentence,category,keyword,marked_sentence,speech_id,speaker,date,quarter,major_heading,minor_heading,speech
0,Friend the Minister this question: what is to ...,[electricity_prices],"[[electricity, bill]]",Friend the Minister this question: what is to ...,uk.org.publicwhip/debate/2014-01-06a.76.2,Anne McIntosh,2014-01-06,2014-Q1,Water Bill,New Clause 3 — Provision of benefits information,I hope that my right hon. Friend will bear wit...
1,Members who have put their names to new clause...,"[electricity_prices, electricity_prices]","[[electricity, bill], [electricity, bills]]",Members who have put their names to new clause...,uk.org.publicwhip/debate/2014-01-06a.76.2,Anne McIntosh,2014-01-06,2014-Q1,Water Bill,New Clause 3 — Provision of benefits information,I hope that my right hon. Friend will bear wit...
2,The size of water bills may not have reached t...,"[electricity_prices, electricity_prices]","[[electricity, bill], [electricity, bills]]",The size of water *bills* may not have reached...,uk.org.publicwhip/debate/2014-01-06a.79.1,Thomas Docherty,2014-01-06,2014-Q1,Water Bill,New Clause 3 — Provision of benefits information,It is always a good thing to be a charitable a...
3,Our reforms will ensure that more than 30% of ...,[electricity_prices],"[[electricity, bill]]",Our reforms will ensure that more than 30% of ...,uk.org.publicwhip/debate/2014-01-08c.283.6,Stephen Crabb,2014-01-08,2014-Q1,WALES,Renewables (Jobs),This Government’s recent announcements on stri...
4,"Wales is a net producer of energy, a major ele...",[electricity_prices],"[[electricity, price]]","Wales is a net producer of energy, a major *el...",uk.org.publicwhip/debate/2014-01-08c.286.3,Albert Owen,2014-01-08,2014-Q1,WALES,Living Standards,"May I associate myself with your words, Mr Spe..."
...,...,...,...,...,...,...,...,...,...,...,...
1815,"While I welcome the news that more than 7,000 ...","[electricity_prices, electricity_prices]","[[electricity, bill], [electricity, bills]]","While I welcome the news that more than 7,000 ...",uk.org.publicwhip/debate/2025-06-23d.876.6,Jim Shannon,2025-06-23,2025-Q2,UK Modern Industrial Strategy,None,"You are very kind, Madam Deputy Speaker—we got..."
1816,I particularly welcome the good news for skill...,"[electricity_prices, electricity_prices]","[[electricity, cost], [electricity, costs]]",I particularly welcome the good news for skill...,uk.org.publicwhip/debate/2025-06-23d.879.2,Melanie Ward,2025-06-23,2025-Q2,UK Modern Industrial Strategy,None,I really warmly welcome the modern industrial ...
1817,"The ISOP will be an expert and impartial body,...","[electricity_prices, electricity_prices]","[[electricity, cost], [electricity, costs]]","The ISOP will be an expert and impartial body,...",uk.org.publicwhip/debate/2024-05-24d.1163.1,Robert Largan,2024-05-24,2024-Q2,Energy,None,"I beg to move, That the draft Energy Act 2023 ..."
1818,"We have made the case, as have significant num...",[electricity_prices],"[[electricity, price]]","We have made the case, as have significant num...",uk.org.publicwhip/debate/2024-05-24d.1164.2,Kirsty Blackman,2024-05-24,2024-Q2,Energy,None,Please accept my apologies for not putting in ...


In [131]:
hit_sentences = (
    _hits_df
    .groupby("speech_id")
    .agg(
        marked_sentence = ("marked_sentence", "; ".join),
        speech = ("speech", lambda x: x.iloc[0]),
    )
    .reset_index()
)

In [132]:
# Construct a list of strings of each speech in the data, with the format: full speech: ..., relevant sentences: ...

list_of_speeches = []

for index, row in hit_sentences.iterrows():
    list_of_speeches.append(f"Relevant sentences: {row.marked_sentence}\n\nFull speech: {row.speech}\n\n")

check_data = dict(zip(hit_sentences.speech_id, list_of_speeches))

In [ ]:
system_message = "Determine whether this speech (1) actually is about electricity prices, and (2) explicitly mentions reducing electricity prices by changing taxation or levies (for example, by moving levies from electricity to gas)"

fields = [
    {"name": "is_about_electricity_prices", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    {"name": "mentions_reducing_electricity_prices", "type": "str", "description": "A one-word answer: 'yes' or 'no'."},
    {"name": "reduction_mechanism", "type": "str", "description": "Proposed mechanism to reduce electricity prices (1-2 word answer); if not proposed, answer 'n/a'"},
]

processor = batch_check.LLMProcessor(
    output_path="output.jsonl",
    system_message=system_message,
    session_name="testing",
    output_fields=fields,
)

processor.run(check_data, batch_size=25, sleep_time=0.5)

2025-08-05 12:23:08,728 - root - INFO - Using OpenAI
2025-08-05 12:23:08,762 - langfuse - WARNING - Langfuse client is disabled since no public_key was provided as a parameter or environment variable 'LANGFUSE_PUBLIC_KEY'. See our docs: https://langfuse.com/docs/sdk/python/low-level-sdk#initialize-client


<Task pending name='Task-23190' coro=<LLMProcessor.process_text_data() running at /Users/karlis.kanders/Code/discovery_mission_radar/.venv/lib/python3.12/site-packages/discovery_utils/utils/llm/batch_check.py:120>>

2025-08-05 12:23:08,795 - root - INFO - Processing batch 1/58
2025-08-05 12:23:11,702 - root - INFO - Processing batch 2/58
2025-08-05 12:23:14,347 - root - INFO - Processing batch 3/58
2025-08-05 12:23:16,526 - root - INFO - Processing batch 4/58
2025-08-05 12:23:19,577 - root - INFO - Processing batch 5/58
2025-08-05 12:23:21,841 - root - INFO - Processing batch 6/58
2025-08-05 12:23:24,766 - root - INFO - Processing batch 7/58
2025-08-05 12:23:27,624 - root - INFO - Processing batch 8/58
2025-08-05 12:23:31,597 - root - INFO - Processing batch 9/58
2025-08-05 12:23:34,864 - root - INFO - Processing batch 10/58
2025-08-05 12:23:38,154 - root - INFO - Processing batch 11/58
2025-08-05 12:23:42,434 - root - INFO - Processing batch 12/58
2025-08-05 12:23:44,975 - root - INFO - Processing batch 13/58
2025-08-05 12:23:47,112 - root - INFO - Processing batch 14/58
2025-08-05 12:23:49,594 - root - INFO - Processing batch 15/58
2025-08-05 12:23:51,788 - root - INFO - Processing batch 16/58
2

In [175]:
checked_speeches_df = (
    pd.read_json("output.jsonl", lines=True)
    .query("id in @_hits_df.speech_id")
)
print(len(checked_speeches_df), len(checked_speeches_df.query("is_about_electricity_prices == 'yes'")))
checked_speeches_df = checked_speeches_df.query("is_about_electricity_prices == 'yes'")

1428 1427


In [176]:
len(checked_speeches_df.query("mentions_reducing_electricity_prices == 'yes'"))

344

In [178]:
import re 

hit_sentences_checked = (
    hit_sentences
    .merge(checked_speeches_df, left_on="speech_id", right_on="id", how="left")
    .merge(_hits_df.drop_duplicates(subset="speech_id")[['speech_id', 'date', 'quarter','speaker', 'major_heading', 'minor_heading']], on="speech_id", how="left")
    .assign(speech_text_norm = lambda df: df.speech.apply(lambda x: re.sub(r"\s+", " ", x)))
    .drop_duplicates(["speaker", "date", "speech_text_norm"])   
)

In [179]:
hit_sentences_checked.head()

,speech_id,marked_sentence,speech,is_about_electricity_prices,mentions_reducing_electricity_prices,reduction_mechanism,id,timestamp,model,temperature,date,quarter,speaker,major_heading,minor_heading,speech_text_norm
0,uk.org.publicwhip/debate/2014-01-06a.76.2,Friend the Minister this question: what is to ...,I hope that my right hon. Friend will bear wit...,yes,no,n/a,uk.org.publicwhip/debate/2014-01-06a.76.2,2025-08-05 11:23:08.797230+00:00,gpt-4o-mini,0.0,2014-01-06,2014-Q1,Anne McIntosh,Water Bill,New Clause 3 — Provision of benefits information,I hope that my right hon. Friend will bear wit...
1,uk.org.publicwhip/debate/2014-01-06a.79.1,The size of water *bills* may not have reached...,It is always a good thing to be a charitable a...,yes,no,n/a,uk.org.publicwhip/debate/2014-01-06a.79.1,2025-08-05 11:23:08.798749+00:00,gpt-4o-mini,0.0,2014-01-06,2014-Q1,Thomas Docherty,Water Bill,New Clause 3 — Provision of benefits information,It is always a good thing to be a charitable a...
2,uk.org.publicwhip/debate/2014-01-08c.283.6,Our reforms will ensure that more than 30% of ...,This Government’s recent announcements on stri...,yes,no,n/a,uk.org.publicwhip/debate/2014-01-08c.283.6,2025-08-05 11:23:08.798979+00:00,gpt-4o-mini,0.0,2014-01-08,2014-Q1,Stephen Crabb,WALES,Renewables (Jobs),This Government’s recent announcements on stri...
3,uk.org.publicwhip/debate/2014-01-08c.286.3,"Wales is a net producer of energy, a major *el...","May I associate myself with your words, Mr Spe...",yes,no,n/a,uk.org.publicwhip/debate/2014-01-08c.286.3,2025-08-05 11:23:08.799162+00:00,gpt-4o-mini,0.0,2014-01-08,2014-Q1,Albert Owen,WALES,Living Standards,"May I associate myself with your words, Mr Spe..."
4,uk.org.publicwhip/debate/2014-01-09b.475.4,"Areas of west and north-west Wales, Pembrokesh...","I would welcome people from Lancashire, the La...",yes,no,n/a,uk.org.publicwhip/debate/2014-01-09b.475.4,2025-08-05 11:23:08.799329+00:00,gpt-4o-mini,0.0,2014-01-09,2014-Q1,Albert Owen,Backbench Business — Rural Communities,None,"I would welcome people from Lancashire, the La..."


In [181]:
# _hit_sentences_checked = hit_sentences_checked.query("mentions_reduce == 'no'")
# i = 12
# print(_hit_sentences_checked.iloc[i].mentions_reduce)
# print(_hit_sentences_checked.iloc[i].reduce_mechanism)
# print("===")
# print(_hit_sentences_checked.iloc[i].marked_sentence)
# print("===")
# print(_hit_sentences_checked.iloc[i].speech)


## Calculate stats

In [182]:
from discovery_utils.utils import (
    analysis_gtr,
    analysis,
    charts,
    google,
    google_slides,
)


In [183]:
from discovery_mission_radar import PROJECT_DIR

PRESENT_QUARTER = "2025-Q2"
OUTPUT_DIR = PROJECT_DIR / "data" / "2025_Q2_ASF" / "hansard"

In [184]:
def impute_missing_quarters(df, date_col="quarter", value_col="speeches", min_quarter=None, max_quarter=None):
    # Convert quarter strings to Period type for easy handling
    df[date_col] = pd.PeriodIndex(df[date_col], freq='Q')
    
    # Determine min and max quarters
    if min_quarter is None:
        min_quarter = df[date_col].min()
    if max_quarter is None:
        max_quarter = df[date_col].max()
    
    # Generate full range of quarters
    full_range = pd.period_range(start=min_quarter, end=max_quarter, freq='Q')
    
    # Create a complete DataFrame with all quarters
    full_df = pd.DataFrame({date_col: full_range})
    
    # Merge with the original DataFrame
    df = full_df.merge(df, on=date_col, how='left')
    
    # Fill missing values in # speeches column with 0
    df[value_col] = df[value_col].fillna(0).astype(int)
    
    # Convert period back to string if necessary
    df[date_col] = df[date_col].astype(str)
    # Add a dash between year and quarter
    df[date_col] = df[date_col].str.replace("Q", "-Q")
    return df

def impute_missing_years(df, year_col="year", value_col="speeches", min_year=None, max_year=None):
    # Ensure year column is integer
    df[year_col] = df[year_col].astype(int)

    # Determine min and max years
    if min_year is None:
        min_year = df[year_col].min()
    if max_year is None:
        max_year = df[year_col].max()

    # Generate a complete range of years
    full_range = pd.DataFrame({year_col: range(min_year, max_year + 1)})

    # Merge with the existing DataFrame
    df = full_range.merge(df, on=year_col, how='left')

    # Fill missing values in # speeches column with 0
    df[value_col] = df[value_col].fillna(0).astype(int)
    return df

In [185]:
def process_data(matching_ids, category_name: str, _speeches_df = hit_sentences_checked, prefix: str = None):

    present_quarter = PRESENT_QUARTER

    selected_df = (
        _speeches_df
        .query("speech_id in @matching_ids")
        .drop_duplicates(subset="speech_id")
        .assign(year = lambda df: df.date.apply(lambda x: x.split("-")[0]))
        # .assign(speech_text_norm = lambda df: df.speech.apply(lambda x: re.sub(r"\s+", " ", x)))
        # .drop_duplicates(["speakername", "date", "speech_text_norm"])        
    )

    # Figure variables
    if prefix is None:
        prefix = f"{OUTPUT_DIR}/charts/hansard_{category_name}_"
    _scale = 2

    ts_quarterly_df = (
        selected_df
        .query("date >= '2023-01-01'")
        .groupby("quarter")
        .agg(speeches = ("speech_id", "count"))
        .reset_index()
        .pipe(impute_missing_quarters, min_quarter="2023Q1", max_quarter="2025Q2")
    )

    fig = charts.ts_bar(
        ts_quarterly_df,
        variable="speeches",
        variable_title="Number of speeches",
        time_column="quarter",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of speeches for {category_name}")
    chart_filename = f"{prefix}quarterly_speeches.png"
    fig.save(chart_filename, scale_factor=_scale)    


    previous_four_quarters = ts_quarterly_df.query("quarter < @present_quarter").sort_values("quarter").tail(4).quarter.tolist()
    previous_four_quarters_mean_df = (
        ts_quarterly_df
        .query("quarter in @previous_four_quarters")
        .assign(_col = "previous_four_quarters")
        .groupby("_col")
        .agg(
            amount=("speeches", "mean"),
        )
        .T
        .reset_index()
        .rename(columns={"index": "variable"})
    )

    present_quarter_df = (
        ts_quarterly_df
        .query("quarter == @present_quarter")
        # rename index to "present_quarter"
        .assign(_col = "magnitude")
        .groupby("_col")
        .agg(
            amount=("speeches", "mean"),
        )
        .T.reset_index().rename(columns={"index": "variable"})
        )

    # growth_magnitude_quarterly_df = (
    #     previous_four_quarters_mean_df
    #     .merge(present_quarter_df, on="variable", how="left")
    #     .assign(growth = lambda df: (df.magnitude - df.previous_four_quarters) / df.previous_four_quarters * 100)
    #     .assign(theme=category_name)
    # )
    growth_magnitude_quarterly_df = pd.DataFrame()

    ts_df = (
        selected_df
        .query("date >= '2014-01-01'")
        .groupby("year")
        .agg(speeches = ("speech_id", "count"))
        .reset_index()
        .pipe(impute_missing_years, min_year=2014, max_year=2025)
    )

    fig = charts.ts_bar(
        ts_df,
        variable="speeches",
        variable_title="Number of speeches",
        time_column="year",
    )
    fig = charts.configure_plots(fig, chart_title=f"Number of speeches for {category_name}")
    chart_filename = f"{prefix}speeches.png"
    fig.save(chart_filename, scale_factor=_scale)        

    growth_magnitude_df = (
        analysis.magnitude_growth(ts_df, year_start=2020, year_end=2024)
        .assign(theme=category_name)
        .reset_index()
        .rename(columns={'index': 'variable'})
    )    

    return ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, selected_df

In [197]:
matching_ids = hit_sentences_checked.speech_id.unique()
category_name = "electricity_prices_reduction"
selected_df = hit_sentences_checked.query("mentions_reducing_electricity_prices == 'yes'")
prefix = None
ts_df, ts_quarterly_df, growth_magnitude_df, growth_magnitude_quarterly_df, selected_df = process_data(matching_ids, category_name, selected_df, prefix)


In [187]:
selected_df

,speech_id,marked_sentence,speech,is_about_electricity_prices,mentions_reducing_electricity_prices,reduction_mechanism,id,timestamp,model,temperature,date,quarter,speaker,major_heading,minor_heading,speech_text_norm,year
0,uk.org.publicwhip/debate/2014-01-06a.76.2,Friend the Minister this question: what is to ...,I hope that my right hon. Friend will bear wit...,yes,no,n/a,uk.org.publicwhip/debate/2014-01-06a.76.2,2025-08-05 11:23:08.797230+00:00,gpt-4o-mini,0.0,2014-01-06,2014-Q1,Anne McIntosh,Water Bill,New Clause 3 — Provision of benefits information,I hope that my right hon. Friend will bear wit...,2014
1,uk.org.publicwhip/debate/2014-01-06a.79.1,The size of water *bills* may not have reached...,It is always a good thing to be a charitable a...,yes,no,n/a,uk.org.publicwhip/debate/2014-01-06a.79.1,2025-08-05 11:23:08.798749+00:00,gpt-4o-mini,0.0,2014-01-06,2014-Q1,Thomas Docherty,Water Bill,New Clause 3 — Provision of benefits information,It is always a good thing to be a charitable a...,2014
2,uk.org.publicwhip/debate/2014-01-08c.283.6,Our reforms will ensure that more than 30% of ...,This Government’s recent announcements on stri...,yes,no,n/a,uk.org.publicwhip/debate/2014-01-08c.283.6,2025-08-05 11:23:08.798979+00:00,gpt-4o-mini,0.0,2014-01-08,2014-Q1,Stephen Crabb,WALES,Renewables (Jobs),This Government’s recent announcements on stri...,2014
3,uk.org.publicwhip/debate/2014-01-08c.286.3,"Wales is a net producer of energy, a major *el...","May I associate myself with your words, Mr Spe...",yes,no,n/a,uk.org.publicwhip/debate/2014-01-08c.286.3,2025-08-05 11:23:08.799162+00:00,gpt-4o-mini,0.0,2014-01-08,2014-Q1,Albert Owen,WALES,Living Standards,"May I associate myself with your words, Mr Spe...",2014
4,uk.org.publicwhip/debate/2014-01-09b.475.4,"Areas of west and north-west Wales, Pembrokesh...","I would welcome people from Lancashire, the La...",yes,no,n/a,uk.org.publicwhip/debate/2014-01-09b.475.4,2025-08-05 11:23:08.799329+00:00,gpt-4o-mini,0.0,2014-01-09,2014-Q1,Albert Owen,Backbench Business — Rural Communities,None,"I would welcome people from Lancashire, the La...",2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1423,uk.org.publicwhip/debate/2025-06-23d.870.0,"I will work as hard as I can to deliver that, ...",I almost thought the hon. Gentleman welcomed p...,yes,no,n/a,uk.org.publicwhip/debate/2025-06-23d.870.0,2025-08-05 11:25:34.459446+00:00,gpt-4o-mini,0.0,2025-06-23,2025-Q2,Jonathan Reynolds,UK Modern Industrial Strategy,None,I almost thought the hon. Gentleman welcomed p...,2025
1424,uk.org.publicwhip/debate/2025-06-23d.873.1,Friends the Members for Stoke-on-Trent North (...,Ceramics UK has described today’s modern indus...,yes,no,n/a,uk.org.publicwhip/debate/2025-06-23d.873.1,2025-08-05 11:25:34.459579+00:00,gpt-4o-mini,0.0,2025-06-23,2025-Q2,Gareth Snell,UK Modern Industrial Strategy,None,Ceramics UK has described today’s modern indus...,2025
1425,uk.org.publicwhip/debate/2025-06-23d.876.6,"While I welcome the news that more than 7,000 ...","You are very kind, Madam Deputy Speaker—we got...",yes,no,n/a,uk.org.publicwhip/debate/2025-06-23d.876.6,2025-08-05 11:25:39.916064+00:00,gpt-4o-mini,0.0,2025-06-23,2025-Q2,Jim Shannon,UK Modern Industrial Strategy,None,"You are very kind, Madam Deputy Speaker—we got...",2025
1426,uk.org.publicwhip/debate/2025-06-23d.879.2,I particularly welcome the good news for skill...,I really warmly welcome the modern industrial ...,yes,no,n/a,uk.org.publicwhip/debate/2025-06-23d.879.2,2025-08-05 11:25:39.916370+00:00,gpt-4o-mini,0.0,2025-06-23,2025-Q2,Melanie Ward,UK Modern Industrial Strategy,None,I really warmly welcome the modern industrial ...,2025


In [194]:
cols = [
    "speech_id",
    "date",
    "year",
    "quarter",
    "major_heading",
    "minor_heading",
    "speaker",
    "marked_sentence",
    "is_about_electricity_prices",
    "mentions_reducing_electricity_prices",
    "reduction_mechanism",
    "speech",
    "model",
    "timestamp",
    "temperature",
]

_selected_df = (
    selected_df[cols]
    .copy()
    # shorten speech to first 3000 characters
    .assign(speech = lambda df: df.speech.str[:3000])
)

In [ ]:
sheet_id = "1XNeeoNBuiBFFQP4X3KRd-vRFZvCCU3QarouCKwawIm4"
google.upload_data_to_gsheet(sheet_id, 
    {
        "speeches": _selected_df,
        "stats": growth_magnitude_df,
        "chart": ts_df,
        "chart_quarterly": ts_quarterly_df,
})
google.format_gsheet(sheet_id, "speeches", freeze_cols=2)
google.format_gsheet(sheet_id, "stats", freeze_cols=0)
google.format_gsheet(sheet_id, "chart", freeze_cols=0)
google.format_gsheet(sheet_id, "chart_quarterly", freeze_cols=0)

2025-08-05 12:38:49,212 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q2 ASF - Hansard
2025-08-05 12:38:52,329 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q2 ASF - Hansard
2025-08-05 12:38:56,033 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q2 ASF - Hansard
2025-08-05 12:39:00,646 - root - INFO - Connected to Google Sheet: Mission Radar 2025 Q2 ASF - Hansard
